In [1]:
import torch
from ebm.util.config import Config
from ebm.networks import get_network
from ebm.networks import DeepHopfieldNetwork
from ebm.estimators.gradient_estimators import EquiPropEstimator
from ebm.estimators.optimizer import SGDOptimizer
from ebm.estimators.cost import SquaredError
from ebm.runner import NetworkRunner

# Create a toy dataset of 2 interlocking rings in 3D and with 1000 points and labels for each point
X = torch.randn(1000, 3)
Y = torch.zeros(1000)
Y[X[:, 0] ** 2 + X[:, 1] ** 2 < 1] = 1
Y[X[:, 0] ** 2 + X[:, 1] ** 2 < 0.5] = 0
Y[X[:, 0] ** 2 + X[:, 1] ** 2 < 0.25] = 1
Y[X[:, 0] ** 2 + X[:, 1] ** 2 < 0.125] = 0
Y = Y.int()
# Plot the dataset
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X[Y == 0, 0], X[Y == 0, 1], X[Y == 0, 2], color='r')
ax.scatter(X[Y == 1, 0], X[Y == 1, 1], X[Y == 1, 2], color='b')
plt.show()

ModuleNotFoundError: No module named 'jaynes'

In [ ]:
# Create a dataloader for the dataset
dataset = torch.utils.data.TensorDataset(X, Y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)
# Split the dataset into a training and test set
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
dataset = 'TwoMoons'
config = Config()
#network = get_network(config)
layers = [2,164,2] #config.layers
network = DeepHopfieldNetwork(layers,config)
cost_fn = SquaredError(network)
optimizer = SGDOptimizer(network)
estimator = EquiPropEstimator(network, cost_fn)
runner = NetworkRunner(network,estimator,optimizer, train_loader, test_loader)
runner.train(num_epochs=100)
runner.eval(custom=False)